# Turbofan RUL Tahmini — Bölüm 5
## Final Model Değerlendirmesi: Resmi Test Seti, Hata Analizi ve Açıklanabilirlik (Aşama 12)

Bu notebook'ta **resmi test seti ilk ve tek kez açılır.** Final model (Aşama 11'de önceden tanımlanmış kuralla seçilen **LightGBM**) 100 test motorunda değerlendirilir; sonuçlar baseline'lar ve literatürle karşılaştırılır, hataları analiz edilir ve model SHAP ile açıklanır.

> 🔒 **Kural:** Bu notebook'taki hiçbir sonuç model seçimini, hiperparametreleri, pencere uzunluğunu veya RUL üst sınırını **değiştirmez.** Test sonucuna bakıp geri dönüp değişiklik yapmak test setini model seçiminde kullanmak (test leakage) olur. Test sonuçları ne çıkarsa çıksın raporlanır ve tartışılır.

> ⏱️ Colab'da ≈ 3–5 dk.

## 0. Kurulum
### 0.1 Drive bağlantısı ve dosya kontrolü
`BTK_RUL` klasöründe bu aşama için iki yeni dosya gerekiyor: **`test_FD001.txt`** ve **`RUL_FD001.txt`**. `final_model.joblib` ve `final_model_meta.json` Bölüm 4 tarafından oluşturuldu.

In [ ]:
import os
BASE_DIR = "/content/drive/MyDrive/BTK_RUL"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir(BASE_DIR)
except ImportError:
    print("Colab dışında çalışılıyor → mevcut klasör kullanılacak.")
print("Çalışma klasörü:", os.getcwd())

required = ["train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt", "final_model.joblib", "final_model_meta.json"]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"Eksik dosyalar: {missing}. test_FD001.txt ve RUL_FD001.txt dosyalarını Drive'daki BTK_RUL klasörüne yükleyin; "
                            "final_model.* dosyaları yoksa önce Bölüm 4'ü çalıştırın.")
print("Tüm dosyalar mevcut ✓")

In [ ]:
%pip install -q shap lightgbm

In [ ]:
%%writefile rul_utils.py
"""rul_utils.py — NASA C-MAPSS RUL projesi için ortak yardımcılar.
Notebook'lar ve Streamlit uygulaması aynı kodu kullanır (tekrarlanabilirlik)."""
import time
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

INDEX_COLS = ["unit", "cycle"]
SETTING_COLS = ["op1", "op2", "op3"]
SENSOR_COLS = [f"s{i}" for i in range(1, 22)]
COLS = INDEX_COLS + SETTING_COLS + SENSOR_COLS
FEATURES_14 = ["s2", "s3", "s4", "s7", "s8", "s9", "s11", "s12",
               "s13", "s14", "s15", "s17", "s20", "s21"]


def load_cmapss(path):
    """Boşlukla ayrılmış, başlıksız C-MAPSS dosyasını okur."""
    df = pd.read_csv(path, sep=r"\s+", header=None).iloc[:, :26]
    df.columns = COLS
    return df.sort_values(["unit", "cycle"]).reset_index(drop=True)


def add_rul(df):
    """Eğitim verisi için gerçek (kırpılmamış) RUL: motorun son döngüsü - mevcut döngü."""
    df = df.copy()
    df["RUL"] = df.groupby("unit")["cycle"].transform("max") - df["cycle"]
    return df


def _rolling_slope(x, window):
    """Geçmiş `window` döngüdeki doğrusal eğim (yalnızca geçmişe bakar)."""
    s = pd.Series(np.asarray(x, dtype=float))
    t = pd.Series(np.arange(len(s), dtype=float))
    cov = s.rolling(window, min_periods=2).cov(t)
    var = t.rolling(window, min_periods=2).var()
    return (cov / var).fillna(0.0).to_numpy()


class RollingFeatures(BaseEstimator, TransformerMixin):
    """Motor bazında, yalnızca GEÇMİŞ döngüleri kullanan kayan pencere özellikleri.

    - Girdi: 'unit', 'cycle' ve sensör sütunlarını içeren DataFrame (motor içinde döngü sırasına göre).
    - Çıktı: ham sensörler (+ isteğe bağlı) kayan ortalama, standart sapma ve eğim.
    - 'unit' ve 'cycle' çıktıya ALINMAZ (özellik değil, yalnızca gruplama/sıra bilgisi).
    - fit() hiçbir şey öğrenmez (stateless): bir satırın özellikleri yalnızca kendi motorunun
      o ana kadarki kayıtlarından hesaplanır → başka motorlardan veya gelecekten bilgi sızmaz.
    """

    def __init__(self, sensors=tuple(FEATURES_14), window=10, stats=("mean", "std", "slope"), keep_raw=True):
        self.sensors = sensors
        self.window = window
        self.stats = stats
        self.keep_raw = keep_raw

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        if not X.groupby("unit")["cycle"].apply(lambda c: c.is_monotonic_increasing).all():
            raise ValueError("Her motorun satırları döngü sırasına göre sıralı olmalı.")
        sensors, w = list(self.sensors), int(self.window)
        g = X.groupby("unit", sort=False)
        parts = []
        if self.keep_raw:
            parts.append(X[sensors].reset_index(drop=True))
        if "mean" in self.stats:
            m = g[sensors].transform(lambda c: c.rolling(w, min_periods=1).mean())
            parts.append(m.add_suffix(f"_mean{w}").reset_index(drop=True))
        if "std" in self.stats:
            sd = g[sensors].transform(lambda c: c.rolling(w, min_periods=2).std()).fillna(0.0)
            parts.append(sd.add_suffix(f"_std{w}").reset_index(drop=True))
        if "slope" in self.stats:
            sl = g[sensors].transform(lambda c: _rolling_slope(c, w))
            parts.append(sl.add_suffix(f"_slope{w}").reset_index(drop=True))
        out = pd.concat(parts, axis=1)
        out.index = X.index
        return out

    def get_feature_names_out(self, input_features=None):
        names = list(self.sensors) if self.keep_raw else []
        for st in ("mean", "std", "slope"):
            if st in self.stats:
                names += [f"{s}_{st}{self.window}" for s in self.sensors]
        return np.array(names)


def nasa_score(y_true, y_pred):
    """PHM08 skor fonksiyonu. d = tahmin - gerçek.
    Geç tahmin (d > 0, ömrü fazla tahmin etmek) daha ağır cezalandırılır."""
    d = np.asarray(y_pred) - np.asarray(y_true)
    return float(np.sum(np.where(d < 0, np.exp(-d / 13.0) - 1, np.exp(d / 10.0) - 1)))


def regression_metrics(y_true, y_pred, prefix=""):
    return {
        f"{prefix}RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        f"{prefix}MAE": float(mean_absolute_error(y_true, y_pred)),
        f"{prefix}R2": float(r2_score(y_true, y_pred)),
        f"{prefix}NASA": nasa_score(y_true, y_pred),
    }


def test_like_rows(df, idx, n_cuts=10, min_cycle=31, max_rul=150, seed=42):
    """Validation motorlarından resmi test protokolünü taklit eden satırlar seçer:
    her motor için rastgele `n_cuts` kesim noktası; kesim anındaki GERÇEK RUL ile değerlendirilir.
    - min_cycle: kayan pencerelerin dolması için gereken minimum geçmiş.
    - max_rul: bakım karar ufku. Kesim noktaları gerçek RUL <= max_rul olan döngülerden seçilir
      (EDA: bozulma ömrün ikinci yarısında görünür hale geliyor; daha erken döngülerde RUL
      sensörlerden ayırt edilemez ve bakım kararı açısından önemsizdir)."""
    rng = np.random.default_rng(seed)
    sub = df.loc[idx]
    chosen = []
    for _, d in sub.groupby("unit"):
        cand = d.index[(d["cycle"] >= min(min_cycle, d["cycle"].max())) & (d["RUL"] <= max_rul)]
        chosen.extend(rng.choice(cand, size=min(n_cuts, len(cand)), replace=False))
    return np.sort(np.array(chosen))


def cv_evaluate(model, df, cap, cv, feature_cols=None, n_cuts=10, max_rul=150, seed=42, name=None,
                train_metrics=False, model_size=False, return_preds=False):
    """Motor bazlı CV ile değerlendirme.
    - Eğitim hedefi: RUL üst sınırla kırpılır (cap=None ise kırpılmaz).
    - 'clip_' metrikleri: validation motorlarının tüm döngüleri, kırpılmış hedefe göre.
    - 'test_' metrikleri: validation motorlarında rastgele kesim noktaları, GERÇEK RUL'a göre
      ve bakım karar ufku (gerçek RUL <= max_rul) içinde → model ve cap seçiminde birincil ölçüt.
    - train_metrics=True: eğitim fold'undaki RMSE (aşırı öğrenme kontrolü).
    - model_size=True: eğitilmiş modelin bellekteki (pickle) boyutu, MB.
    - return_preds=True: (sonuç_tablosu, kesim_noktası_tahminleri) döndürür (hata analizi için).
    """
    feature_cols = feature_cols or (INDEX_COLS + FEATURES_14)
    X = df[feature_cols]
    y_true = df["RUL"].to_numpy()
    y_fit = y_true if cap is None else np.minimum(y_true, cap)
    rows, preds = [], []
    for k, (tr, va) in enumerate(cv.split(X, y_fit, groups=df["unit"])):
        m = clone(model)
        t0 = time.perf_counter()
        m.fit(X.iloc[tr], y_fit[tr])
        fit_s = time.perf_counter() - t0
        t0 = time.perf_counter()
        p = np.clip(m.predict(X.iloc[va]), 0, None)
        pred_s = time.perf_counter() - t0
        res = {"fold": k + 1, "fit_s": fit_s, "pred_s": pred_s}
        res.update(regression_metrics(y_fit[va], p, "clip_"))
        cut = test_like_rows(df, df.index[va], n_cuts=n_cuts, max_rul=max_rul, seed=seed + k)
        pos = pd.Series(np.arange(len(va)), index=df.index[va]).loc[cut].to_numpy()
        res.update(regression_metrics(y_true[cut], p[pos], "test_"))
        # Arızaya yakın bölge (gerçek RUL <= 50): bakım kararının en kritik olduğu dönem
        near = y_true[cut] <= 50
        res["near50_RMSE"] = float(np.sqrt(np.mean((p[pos][near] - y_true[cut][near]) ** 2))) if near.any() else np.nan
        # Geç tahmin oranı: ömrü 10 döngüden fazla FAZLA tahmin etme (güvenlik açısından tehlikeli)
        res["late10_pct"] = float(np.mean(p[pos] - y_true[cut] > 10) * 100)
        if train_metrics:
            p_tr = np.clip(m.predict(X.iloc[tr]), 0, None)
            res["train_clip_RMSE"] = float(np.sqrt(mean_squared_error(y_fit[tr], p_tr)))
        if model_size:
            import pickle
            res["size_MB"] = len(pickle.dumps(m)) / 1e6
        rows.append(res)
        if return_preds:
            preds.append(pd.DataFrame({"model": name, "fold": k + 1, "unit": df.loc[cut, "unit"].to_numpy(),
                                       "cycle": df.loc[cut, "cycle"].to_numpy(),
                                       "RUL_true": y_true[cut], "RUL_pred": p[pos]}))
    out = pd.DataFrame(rows)
    if name is not None:
        out.insert(0, "model", name)
    if return_preds:
        return out, pd.concat(preds, ignore_index=True)
    return out


def summarize(cv_df, by="model"):
    cols = [c for c in ["test_RMSE", "test_MAE", "test_R2", "test_NASA", "near50_RMSE", "late10_pct",
                        "clip_RMSE", "train_clip_RMSE", "fit_s", "pred_s", "size_MB"] if c in cv_df.columns]
    agg = cv_df.groupby(by)[cols].agg(["mean", "std"])
    agg.columns = [f"{a}_{b}" for a, b in agg.columns]
    return agg.sort_values("test_RMSE_mean")


def oof_predictions(model, df, cap, cv, feature_cols=None):
    """Out-of-fold tahminler: her satır, motoru validation'dayken eğitilen modelden tahmin edilir."""
    feature_cols = feature_cols or (INDEX_COLS + FEATURES_14)
    X = df[feature_cols]
    y_fit = df["RUL"].to_numpy() if cap is None else np.minimum(df["RUL"].to_numpy(), cap)
    oof = np.zeros(len(df))
    for tr, va in cv.split(X, y_fit, groups=df["unit"]):
        m = clone(model).fit(X.iloc[tr], y_fit[tr])
        oof[va] = np.clip(m.predict(X.iloc[va]), 0, None)
    return oof

In [ ]:
import warnings, json, time, platform, joblib
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import confusion_matrix, classification_report
from rul_utils import *

sns.set_theme(style="whitegrid", palette="tab10")
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)
print("Python", platform.python_version(), "| scikit-learn", sklearn.__version__)

### 0.2 Final modelin ve kararların yüklenmesi

In [ ]:
meta = json.load(open("final_model_meta.json"))
final_pipe = joblib.load("final_model.joblib")
CAP, WINDOW, SEED = meta["cap"], meta["window"], 42
print(f"Final model : {meta['final_model']}")
print(f"Pencere     : {WINDOW} | RUL üst sınırı: {CAP}")
print(f"CV (bağımsız yeniden değerlendirme) RMSE: {meta['cv_test_RMSE']:.2f} ± {meta['cv_test_RMSE_std']:.2f} | NASA: {meta['cv_test_NASA']:.0f}")
print("Hiperparametreler:")
for k, v in meta["params"].items():
    print(f"   {k:18s} = {v}")

### 0.3 Bölüm 4 sonuçlarının özeti (rapor için)
Bölüm 4'ün kaydettiği dosyalardan, final kararın dayandığı tablo ve **seçim yanlılığı** analizi. Bu analiz, aramadaki en iyi skor ile aramada kullanılmamış noktalardaki skor arasındaki farkı gösteriyor.

In [ ]:
if os.path.exists("results_stage11_summary.csv"):
    s11 = pd.read_csv("results_stage11_summary.csv", index_col=0)
    show = s11[["test_RMSE_mean", "test_RMSE_std", "test_NASA_mean", "near50_RMSE_mean", "late10_pct_mean",
                "fit_s_mean", "size_MB_mean", "tamamlanan_deneme", "seçilen_pencere"]]
    display(show.round(2))
    if os.path.exists("results_stage10_11_trials.csv"):
        tr = pd.read_csv("results_stage10_11_trials.csv")
        best_search = tr[tr["study"].isin(s11.index)].groupby("study")["value"].min()
        sb = pd.DataFrame({"arama_en_iyi_RMSE": best_search, "bağımsız_RMSE": s11["test_RMSE_mean"]}).dropna()
        sb["iyimserlik_farkı"] = sb["bağımsız_RMSE"] - sb["arama_en_iyi_RMSE"]
        print("\nSeçim yanlılığı: aramadaki en iyi skor vs aramada kullanılmamış noktalardaki skor")
        display(sb.sort_values("bağımsız_RMSE").round(3))
else:
    print("results_stage11_summary.csv bulunamadı (Bölüm 4 çıktısı) — bu özet atlandı.")

**Yorum:** Aramadaki en iyi skorlar, aramada kullanılmamış noktalarda **0.2–0.6 RMSE** kötüleşti. Bu, 30 deneme arasından "en iyi"yi seçmenin doğal iyimserliği. Bölüm 4'te final kararı bağımsız skorlara dayandırmamız tam bu yüzden önemliydi. Bağımsız skorlarda Random Forest, XGBoost, HGB ve LightGBM birbirinden **0.4 RMSE** içinde, yani istatistiksel olarak eşdeğer. Önceden tanımlanmış kural bu durumda en küçük ve en hızlı modeli (LightGBM: ~0.8 MB, fold başına ~3 sn eğitim) seçti. Random Forest ~38 kat büyük ve ~12 kat yavaş.

## Aşama 12 — Final Model Değerlendirmesi
### 12.0 Test setinin açılması

Resmi test protokolü (literatürle aynı):
- 100 test motorunun her biri ömrünün rastgele bir noktasında kesilmiş.
- Model her motorun **o ana kadarki tüm kaydını** görür ve **yalnızca son döngü** için kalan ömür tahmin eder.
- Tahmin, `RUL_FD001.txt`'deki **gerçek (kırpılmamış) RUL** ile karşılaştırılır.

In [ ]:
train = add_rul(load_cmapss("train_FD001.txt"))
test = load_cmapss("test_FD001.txt")
rul_true = pd.read_csv("RUL_FD001.txt", sep=r"\s+", header=None).iloc[:, 0].to_numpy()

hist_len = test.groupby("unit")["cycle"].max()
print(f"Test: {test.shape[0]} satır, {test['unit'].nunique()} motor | gerçek RUL dosyası: {len(rul_true)} değer")
assert test["unit"].nunique() == len(rul_true) == 100, "Motor sayısı ile RUL dosyası uyuşmuyor!"
print(f"Test motorlarının geçmiş uzunluğu: min {hist_len.min()}, medyan {hist_len.median():.0f}, max {hist_len.max()} döngü")
print(f"Gerçek RUL: min {rul_true.min()}, medyan {np.median(rul_true):.0f}, max {rul_true.max()} | RUL > {CAP} olan motor: {(rul_true > CAP).sum()}")
print(f"Pencereden ({WINDOW}) kısa geçmişi olan motor sayısı: {(hist_len < WINDOW).sum()}")

fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
sns.histplot(rul_true, bins=20, ax=ax[0]); ax[0].axvline(CAP, c="r", ls="--", label=f"cap={CAP}"); ax[0].set_title("Test motorlarının gerçek RUL dağılımı"); ax[0].legend()
sns.histplot(hist_len, bins=20, ax=ax[1], color="tab:orange"); ax[1].axvline(WINDOW, c="r", ls="--", label=f"pencere={WINDOW}"); ax[1].set_title("Test motorlarının geçmiş uzunluğu (döngü)"); ax[1].legend()
plt.tight_layout(); plt.show()

### 12.1 Test seti metrikleri — final model ve baseline'lar

Karşılaştırma için baseline'lar ve Bölüm 3'ün varsayılan adayı **tüm eğitim verisiyle** yeniden eğitilip aynı test setinde değerlendirilir. Bu modeller yalnızca **referans** içindir; final model seçimini değiştirmez.

| Metrik | Açıklama |
|---|---|
| RMSE, MAE, R² | Gerçek RUL'a göre (literatürle aynı) |
| NASA skoru | 100 motor üzerinden **toplam** PHM08 skoru (literatür bu şekilde raporlar); geç tahmin üstel cezalandırılır |
| RMSE (RUL ≤ 50) | Arızaya yakın motorlar — bakım kararının kritik olduğu bölge |
| Geç tahmin % | Ömrü 10 döngüden fazla **fazla** tahmin edilen motor oranı (güvenlik riski) |

In [ ]:
X_tr, y_tr = train[INDEX_COLS + FEATURES_14], train["RUL"].clip(upper=CAP)
X_te = test[INDEX_COLS + FEATURES_14]
last_pos = test.groupby("unit").tail(1).index.to_numpy()     # her motorun son döngüsü

def ttr(estimator, window=30, stats=("mean", "std", "slope")):
    return TransformedTargetRegressor(regressor=Pipeline([("features", RollingFeatures(window=window, stats=stats)),
                                                          ("scaler", StandardScaler()), ("model", estimator)]),
                                      transformer=StandardScaler())

refs = {
    "Dummy (ortalama)": ttr(DummyRegressor(strategy="mean"), stats=()),
    "Baseline: LinearRegression (ham sensör)": ttr(LinearRegression(), stats=()),
    "Bölüm 3 adayı: HGB varsayılan (w=30)": ttr(HistGradientBoostingRegressor(random_state=SEED), window=30),
}
PRED = {}
for name, m in refs.items():
    m.fit(X_tr, y_tr)
    PRED[name] = np.clip(m.predict(X_te), 0, None)[last_pos]

t0 = time.perf_counter()
pred_all = np.clip(final_pipe.predict(X_te), 0, None)
PRED[f"FINAL: {meta['final_model']} (w={WINDOW})"] = pred_all[last_pos]
print(f"Final model 100 motor için {len(X_te)} döngüyü {time.perf_counter()-t0:.2f} sn'de işledi.")
FINAL_KEY = f"FINAL: {meta['final_model']} (w={WINDOW})"

def test_metrics(y, p):
    m = regression_metrics(y, p)
    near = y <= 50
    m["RMSE (RUL≤50)"] = float(np.sqrt(np.mean((p[near] - y[near]) ** 2)))
    m["geç tahmin %"] = float(np.mean(p - y > 10) * 100)
    m["RMSE (gerçek RUL 125'te kırpılmış)"] = float(np.sqrt(np.mean((p - np.minimum(y, CAP)) ** 2)))
    return m

test_table = pd.DataFrame({n: test_metrics(rul_true, p) for n, p in PRED.items()}).T
test_table.to_csv("results_stage12_test_metrics.csv")
display(test_table.round(2))
fm = test_table.loc[FINAL_KEY]
bl = test_table.loc["Baseline: LinearRegression (ham sensör)"]
print(f"\nFinal model test RMSE: {fm['RMSE']:.2f} | baseline'a göre iyileşme: %{(bl['RMSE'] - fm['RMSE']) / bl['RMSE'] * 100:.1f}")

**Not — "kırpılmış" RMSE sütunu:** Literatürdeki bazı çalışmalar test setinin gerçek RUL değerlerini de 125'te kırparak raporlar. Gerçek RUL'u 125'in üzerinde olan birkaç test motorunda modelimiz tasarım gereği ~125 tahmin eder; bu sütun bu motorların etkisini gösterir. Ana sonuç olarak **kırpılmamış** gerçek RUL'a göre RMSE raporlanır (daha zor ve daha dürüst olan ölçüt).

### 12.2 Genelleme: CV tahmini ile test sonucu tutarlı mı?

In [ ]:
cv_m, cv_s = meta["cv_test_RMSE"], meta["cv_test_RMSE_std"]
t_rmse = fm["RMSE"]
print(f"CV (5 fold, bağımsız noktalar): {cv_m:.2f} ± {cv_s:.2f}   →   aralık ≈ [{cv_m - 2*cv_s:.2f}, {cv_m + 2*cv_s:.2f}] (± 2 std)")
print(f"Resmi test                     : {t_rmse:.2f}")
inside = cv_m - 2 * cv_s <= t_rmse <= cv_m + 2 * cv_s
print("Test sonucu CV'nin ±2 std aralığında mı?", "EVET → validation protokolü test performansını iyi öngördü" if inside
      else "HAYIR → aşağıdaki yorum bölümüne bakın")

# Bootstrap: 100 motorluk test setinde RMSE'nin belirsizliği
rng = np.random.default_rng(SEED)
boots = [np.sqrt(np.mean((PRED[FINAL_KEY][i] - rul_true[i]) ** 2)) for i in (rng.integers(0, 100, 100) for _ in range(2000))]
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"Test RMSE %95 bootstrap güven aralığı: [{lo:.2f}, {hi:.2f}]")

**Neden bootstrap?** Test seti yalnızca 100 motordan oluşuyor; birkaç motorun hatası RMSE'yi belirgin şekilde oynatabilir. Motorları yerine koyarak 2000 kez yeniden örnekleyip RMSE'nin dağılımına bakmak, raporlanan tek sayının ne kadar kesin olduğunu gösterir. Literatürle karşılaştırırken bu aralık akılda tutulmalı.

> 📝 **Rapor için:** CV sonucu, test sonucu ve güven aralığını birlikte yazın. Test RMSE'si CV'den biraz farklıysa bunun normal olduğunu (100 motor, farklı kesim noktası dağılımı) güven aralığıyla gösterin.

### 12.3 Tahmin – gerçek karşılaştırması

In [ ]:
res = pd.DataFrame({"unit": np.arange(1, 101), "RUL_true": rul_true, "RUL_pred": PRED[FINAL_KEY],
                    "baseline_pred": PRED["Baseline: LinearRegression (ham sensör)"], "geçmiş_döngü": hist_len.to_numpy()})
res["hata"] = res["RUL_pred"] - res["RUL_true"]
res["mutlak_hata"] = res["hata"].abs()

fig, ax = plt.subplots(1, 3, figsize=(21, 5.5))
ax[0].scatter(res["RUL_true"], res["baseline_pred"], alpha=.5, s=25, c="gray", label="baseline (LR)")
ax[0].scatter(res["RUL_true"], res["RUL_pred"], alpha=.8, s=30, label=meta["final_model"])
mx = max(res["RUL_true"].max(), 150)
ax[0].plot([0, mx], [0, mx], "k--", lw=1); ax[0].plot([0, CAP, mx], [0, CAP, CAP], "r:", lw=1.5, label="model tavanı (cap)")
ax[0].set_xlabel("Gerçek RUL"); ax[0].set_ylabel("Tahmin"); ax[0].set_title("Tahmin vs gerçek (100 test motoru)"); ax[0].legend()

ax[1].scatter(res["RUL_true"], res["hata"], c=np.where(res["hata"] > 10, "tab:red", "tab:blue"), s=30, alpha=.8)
ax[1].axhline(0, c="k", lw=1); ax[1].axhspan(10, res["hata"].max() + 5, color="red", alpha=.06)
ax[1].set_xlabel("Gerçek RUL"); ax[1].set_ylabel("Hata = tahmin − gerçek"); ax[1].set_title("Residual (kırmızı bölge: >10 döngü geç tahmin)")

o = res.sort_values("RUL_true").reset_index(drop=True)
ax[2].plot(o.index, o["RUL_true"], "k-", lw=2, label="gerçek")
ax[2].plot(o.index, o["RUL_pred"], "o", ms=4, label=meta["final_model"])
ax[2].plot(o.index, o["baseline_pred"], "x", ms=4, c="gray", alpha=.6, label="baseline")
ax[2].set_xlabel("Motorlar (gerçek RUL'a göre sıralı)"); ax[2].set_ylabel("RUL"); ax[2].set_title("Motor bazında tahminler"); ax[2].legend()
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 3.5))
sns.histplot(res["hata"], bins=25, kde=True, ax=ax); ax.axvline(0, c="k")
ax.set_title(f"Hata dağılımı — ortalama {res['hata'].mean():+.1f}, medyan {res['hata'].median():+.1f} döngü"); plt.show()

> 📝 **Rapor için yorumlanacaklar:**
> - Noktalar köşegene ne kadar yakın? Baseline (gri) ile final model arasındaki fark özellikle hangi RUL aralığında belirgin?
> - Ortalama hata pozitif mi negatif mi? Negatifse model genel olarak **temkinli** (ömrü eksik tahmin ediyor) → güvenlik açısından olumlu.
> - Gerçek RUL'u 125'in üzerinde olan motorlarda tahminlerin ~125'te yığılması **beklenen** davranış (Bölüm 2, 5.1).

### 12.4 Karar seviyesinde değerlendirme: risk sınıfları

Uygulamada mühendis RUL sayısından çok **"ne yapmalıyım?"** sorusunun cevabıyla ilgilenir. Regresyon çıktısı örnek bir iş kuralıyla üç risk sınıfına çevrilip **confusion matrix** ile değerlendirilir:

| Risk sınıfı | RUL | Aksiyon |
|---|---|---|
| 🔴 Kritik | ≤ 30 | Motoru bakıma al |
| 🟠 Bakım planla | 31 – 75 | Bakımı takvime ekle |
| 🟢 Normal | > 75 | Rutin izleme |

En tehlikeli hata: gerçekte **Kritik** olan bir motoru "Normal" ya da "Bakım planla" olarak görmek.

In [ ]:
CRIT, PLAN = 30, 75
LABELS = ["Kritik", "Bakım planla", "Normal"]
to_risk = lambda r: np.where(r <= CRIT, "Kritik", np.where(r <= PLAN, "Bakım planla", "Normal"))
res["risk_gerçek"], res["risk_tahmin"] = to_risk(res["RUL_true"]), to_risk(res["RUL_pred"])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, (name, col) in zip(axes, [("Baseline (LR)", "baseline_pred"), (meta["final_model"], "RUL_pred")]):
    cm = confusion_matrix(res["risk_gerçek"], to_risk(res[col]), labels=LABELS)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS, ax=ax, cbar=False)
    ax.set_xlabel("Tahmin edilen risk"); ax.set_ylabel("Gerçek risk"); ax.set_title(name)
plt.tight_layout(); plt.show()
print(classification_report(res["risk_gerçek"], res["risk_tahmin"], labels=LABELS, digits=3, zero_division=0))
crit = res[res["risk_gerçek"] == "Kritik"]
missed = crit[crit["risk_tahmin"] != "Kritik"]
print(f"Gerçekte kritik {len(crit)} motorun {len(crit) - len(missed)} tanesi doğru yakalandı; kaçırılan: {len(missed)}")
if len(missed):
    display(missed[["unit", "RUL_true", "RUL_pred", "hata", "risk_tahmin", "geçmiş_döngü"]].round(1))

> 📝 **Rapor için:** Kritik sınıfın **recall** değeri (gerçekte kritik olanların ne kadarı yakalandı) güvenlik açısından en önemli sayıdır. Kaçırılan kritik motor varsa tahmin edilen RUL'larının eşiğe ne kadar yakın olduğuna bakın (ör. gerçek 28, tahmin 33 → sınırda bir hata; gerçek 10, tahmin 80 → ciddi bir hata). Eşikler bir iş kuralıdır, modelin parçası değildir; arayüzde kullanıcı tarafından değiştirilebilecek.

### 12.5 Hata analizi: model nerede zorlanıyor?

In [ ]:
print("En büyük mutlak hataya sahip 10 motor:")
worst = res.sort_values("mutlak_hata", ascending=False).head(10)
display(worst[["unit", "RUL_true", "RUL_pred", "hata", "geçmiş_döngü", "risk_gerçek", "risk_tahmin"]].round(1))

fig, ax = plt.subplots(1, 2, figsize=(16, 4.8))
ax[0].scatter(res["geçmiş_döngü"], res["mutlak_hata"], s=30, alpha=.7)
ax[0].axvline(WINDOW, c="r", ls="--", label=f"pencere={WINDOW}")
ax[0].set_xlabel("Test motorunun geçmiş uzunluğu (döngü)"); ax[0].set_ylabel("|hata|"); ax[0].set_title("Geçmiş uzunluğu ile hata"); ax[0].legend()
grp = pd.cut(res["RUL_true"], [0, 25, 50, 75, 100, 125, 200], include_lowest=True)
g = res.groupby(grp, observed=True).agg(motor=("unit", "count"), RMSE=("hata", lambda e: np.sqrt(np.mean(e**2))), ort_hata=("hata", "mean"))
g["RMSE"].plot(kind="bar", ax=ax[1], color="tab:blue", alpha=.8); ax[1].set_title("Gerçek RUL aralığına göre RMSE"); ax[1].set_xlabel("gerçek RUL"); ax[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
display(g.round(2))

short = res["geçmiş_döngü"] < WINDOW
print(f"Geçmişi pencereden kısa motorlar ({short.sum()} adet) RMSE: {np.sqrt(np.mean(res.loc[short, 'hata']**2)):.2f} | "
      f"diğerleri ({(~short).sum()} adet): {np.sqrt(np.mean(res.loc[~short, 'hata']**2)):.2f}")

In [ ]:
# En kötü tahmin edilen motorun sensör geçmişi: neden zorlandı?
u_bad = int(worst.iloc[0]["unit"])
u_good = int(res[(res["RUL_true"] - res.loc[res.unit == u_bad, "RUL_true"].iloc[0]).abs() < 15].sort_values("mutlak_hata").iloc[0]["unit"])
key = ["s4", "s11", "s12", "s21"]
fig, axes = plt.subplots(1, 4, figsize=(22, 4))
for ax, s in zip(axes, key):
    for u, lab in [(u_bad, f"motor {u_bad} (en büyük hata)"), (u_good, f"motor {u_good} (benzer gerçek RUL, düşük hata)")]:
        d = test[test["unit"] == u]
        ax.plot(d["cycle"], d[s], lw=1, label=lab)
    ax.set_title(s); ax.set_xlabel("döngü")
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()
print(res[res.unit.isin([u_bad, u_good])][["unit", "RUL_true", "RUL_pred", "hata", "geçmiş_döngü"]].round(1).to_string(index=False))

> 📝 **Rapor için:** En büyük hataların ortak özelliği nedir? Tipik nedenler:
> 1. **Gerçek RUL > 125:** Model tasarım gereği ~125 tahmin eder; bu hata temkinli tarafta ve zararsızdır.
> 2. **Kısa geçmiş:** Bozulma trendi henüz görünmüyor (eğim özellikleri belirsiz).
> 3. **Atipik motor:** Sensörleri eğitimdeki motorlardan farklı seyrediyor (yukarıdaki grafik).
>
> Hangi motorların hangi nedene girdiğini tabloya bakarak yazın.

### 12.6 Sağlamlık (robustness) testleri

İki gerçek hayat senaryosu test edilir:
1. **Sensör gürültüsü artarsa?** Test sensörlerine, her sensörün eğitimdeki standart sapmasının belirli bir oranında Gauss gürültüsü eklenir (5 tekrar ortalaması).
2. **Kısa geçmişle tahmin:** Arayüzde kullanıcı motorun tüm geçmişini değil, yalnızca son N döngüsünü yükleyebilir. Her test motorunun yalnızca son N döngüsü kullanılır.

In [ ]:
sd = train[FEATURES_14].std()
rows = []
for k in [0.0, 0.05, 0.1, 0.2, 0.3]:
    vals = []
    for rep in range(5 if k > 0 else 1):
        noisy = test.copy()
        rng = np.random.default_rng(100 + rep)
        noisy[FEATURES_14] = noisy[FEATURES_14] + rng.normal(0, 1, (len(noisy), len(FEATURES_14))) * (k * sd.to_numpy())
        p = np.clip(final_pipe.predict(noisy[INDEX_COLS + FEATURES_14]), 0, None)[last_pos]
        vals.append(np.sqrt(np.mean((p - rul_true) ** 2)))
    rows.append({"senaryo": "gürültü", "seviye": f"{k:.2f} × std", "RMSE": np.mean(vals)})

for n in [None, 100, 75, WINDOW, 31]:
    sub = test if n is None else test.groupby("unit", group_keys=False).tail(n).reset_index(drop=True)
    lp = sub.groupby("unit").tail(1).index.to_numpy()
    p = np.clip(final_pipe.predict(sub[INDEX_COLS + FEATURES_14]), 0, None)[lp]
    rows.append({"senaryo": "geçmiş uzunluğu", "seviye": "tüm geçmiş" if n is None else f"son {n} döngü", "RMSE": np.sqrt(np.mean((p - rul_true) ** 2))})

rob = pd.DataFrame(rows)
display(rob.round(2))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for a, (s, g) in zip(ax, rob.groupby("senaryo", sort=False)):
    a.plot(g["seviye"], g["RMSE"], marker="o"); a.set_title(s); a.set_ylabel("Test RMSE"); a.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

**Not:** Kısa geçmiş senaryosunda yalnızca son N döngü kullanılıyor ve `cycle` değerleri değişmiyor. Model `cycle` bilgisini zaten kullanmadığı için (Bölüm 2, 5.2) bu senaryo arayüzdeki gerçek kullanımı doğru temsil ediyor.

> 📝 **Rapor için:** Gürültü 0.1 × std'ye çıktığında RMSE ne kadar artıyor? Arayüzde önerilecek **minimum geçmiş uzunluğu** ne olmalı (RMSE'nin belirgin şekilde bozulmaya başladığı N)? Bu sayı Aşama 13'te arayüzdeki "düşük güven" uyarısının eşiği olacak.

### 12.7 Açıklanabilirlik: model kararlarını neye dayandırıyor? (SHAP)

SHAP, her tahmini özelliklerin katkılarına ayırır: *"Bu motorun tahmini RUL'u ortalamadan 40 döngü düşük, bunun 15 döngüsü s11'in son 50 döngüdeki ortalamasından geliyor."* Model hedefi standartlaştırılmış ölçekte öğrendiği için SHAP değerleri **döngü** birimine geri çevrilir.

In [ ]:
reg = final_pipe.regressor_
fe, sc, mdl = reg.named_steps["features"], reg.named_steps["scaler"], reg.named_steps["model"]
feat_names = list(fe.get_feature_names_out())
F_test = fe.transform(X_te)
X_last = pd.DataFrame(sc.transform(F_test)[last_pos], columns=feat_names)
F_last = F_test.iloc[last_pos].reset_index(drop=True)
y_scale, y_mean = float(final_pipe.transformer_.scale_[0]), float(final_pipe.transformer_.mean_[0])

SHAP_OK = False
try:
    import shap
    explainer = shap.TreeExplainer(mdl)
    sv = np.asarray(explainer.shap_values(X_last)) * y_scale                       # döngü birimi
    base = float(np.ravel(explainer.expected_value)[0]) * y_scale + y_mean
    check = np.abs(base + sv.sum(1) - final_pipe.predict(X_te)[last_pos]).max()
    print(f"SHAP toplamsallık kontrolü (taban + katkılar = tahmin): maks fark {check:.4f} döngü")
    imp = pd.Series(np.abs(sv).mean(0), index=feat_names).sort_values(ascending=False)
    SHAP_OK = True
except Exception as e:
    print("SHAP kullanılamadı → permutation importance'a geçiliyor:", repr(e)[:150])
    from sklearn.inspection import permutation_importance
    samp = train.sample(4000, random_state=SEED).sort_values(["unit", "cycle"]).index
    Xs = pd.DataFrame(sc.transform(fe.transform(X_tr))[samp], columns=feat_names)
    ys = (y_tr.to_numpy()[samp] - y_mean) / y_scale
    pi = permutation_importance(mdl, Xs, ys, n_repeats=5, random_state=SEED, scoring="neg_root_mean_squared_error")
    imp = pd.Series(pi.importances_mean * y_scale, index=feat_names).sort_values(ascending=False)

def split_name(f):
    parts = f.split("_")
    return parts[0], ("ham" if len(parts) == 1 else "".join(ch for ch in parts[1] if ch.isalpha()))
grp = pd.DataFrame([(f, *split_name(f), v) for f, v in imp.items()], columns=["özellik", "sensör", "istatistik", "önem"])
by_sensor = grp.groupby("sensör")["önem"].sum().sort_values(ascending=False)
by_stat = grp.groupby("istatistik")["önem"].sum().sort_values(ascending=False)
sym = {"s2": "T24", "s3": "T30", "s4": "T50", "s7": "P30", "s8": "Nf", "s9": "Nc", "s11": "Ps30", "s12": "phi",
       "s13": "NRf", "s14": "NRc", "s15": "BPR", "s17": "htBleed", "s20": "W31", "s21": "W32"}

fig, ax = plt.subplots(1, 3, figsize=(22, 5.5))
imp.head(15)[::-1].plot(kind="barh", ax=ax[0], color="tab:blue"); ax[0].set_title("En önemli 15 özellik" + (" (ort. |SHAP|, döngü)" if SHAP_OK else " (permutation)"))
by_sensor.rename(lambda s: f"{s} ({sym.get(s, '')})")[::-1].plot(kind="barh", ax=ax[1], color="tab:green"); ax[1].set_title("Sensör bazında toplam önem")
(by_stat / by_stat.sum() * 100).plot(kind="bar", ax=ax[2], color="tab:orange"); ax[2].set_title("Özellik türüne göre önem payı (%)"); ax[2].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
display((by_stat / by_stat.sum() * 100).round(1).rename("pay_%").to_frame())

In [ ]:
if SHAP_OK:
    shap.summary_plot(sv, F_last, feature_names=feat_names, max_display=15, show=False)
    plt.title("SHAP özet grafiği — 100 test motoru (renk: özellik değeri)"); plt.tight_layout(); plt.show()

    i = int(res.sort_values("RUL_true").index[0])     # en kritik test motoru
    exp = shap.Explanation(values=sv[i], base_values=base, data=F_last.iloc[i].to_numpy(), feature_names=feat_names)
    shap.plots.waterfall(exp, max_display=12, show=False)
    plt.title(f"Motor {res.loc[i, 'unit']}: gerçek RUL {res.loc[i, 'RUL_true']}, tahmin {res.loc[i, 'RUL_pred']:.1f}")
    plt.tight_layout(); plt.show()
else:
    print("SHAP grafikleri atlandı (SHAP kurulamadı).")

> 📝 **Rapor için:**
> - En etkili sensörler Bölüm 1'deki EDA bulgularıyla (s11, s4, s12, s7, s21 korelasyonu en yüksek sensörlerdi) tutarlı mı?
> - Özellik türü grafiği, feature engineering'in değerini gösteriyor. Kayan ortalama/eğim özelliklerinin toplam payı ham sensörlerden büyükse, gürültüden arındırılmış trendlerin modelin ana bilgi kaynağı olduğunu gösterir.
> - Özet grafikte renk yönü: örneğin `s11_mean50` yüksekken (kırmızı) SHAP negatifse → "HPC statik basıncı arttıkça kalan ömür azalıyor" → fiziksel olarak anlamlı.
> - Waterfall grafiği tek bir motorun tahminini adım adım açıklar; sunumda kullanmak için çok uygun.

### 12.8 Literatürle karşılaştırma (FD001 resmi test seti, RMSE)

Aşağıdaki değerler yayımlanmış çalışmalardan alınmıştır (proje sürecinde yapılan literatür taramasından). **Rapora koymadan önce her kaynağı orijinal makaleden doğrulayın.** Çalışmaların ön işleme ayrıntıları (RUL üst sınırı, pencere boyutu, sensör seçimi) birbirinden farklı olabilir; karşılaştırma bu nedenle yaklaşık niteliktedir.

In [ ]:
lit = pd.DataFrame([
    ("Linear Regression (bu proje, baseline)", "Klasik ML", test_table.loc["Baseline: LinearRegression (ham sensör)", "RMSE"]),
    ("CNN — Babu vd. (2016)", "Derin öğrenme", 18.45),
    ("LSTM — Zheng vd. (2017)", "Derin öğrenme", 16.14),
    ("Tek katmanlı LSTM — karşılaştırmalı çalışma (2026)", "Derin öğrenme", 14.93),
    ("DCNN — Li vd. (2018)", "Derin öğrenme", 12.61),
    ("DA-CNN", "Derin öğrenme", 11.78),
    ("MS-DCNN", "Derin öğrenme", 11.44),
    ("DCGAN", "Derin öğrenme", 10.71),
    (f"{meta['final_model']} + kayan özellikler (bu proje, FINAL)", "Klasik ML", fm["RMSE"]),
], columns=["yöntem", "tür", "RMSE"]).sort_values("RMSE", ascending=False)
display(lit.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["tab:red" if "FINAL" in y else ("tab:gray" if "bu proje" in y else "tab:blue") for y in lit["yöntem"]]
ax.barh(lit["yöntem"], lit["RMSE"], color=colors)
ax.errorbar(fm["RMSE"], list(lit["yöntem"]).index([y for y in lit["yöntem"] if "FINAL" in y][0]),
            xerr=[[fm["RMSE"] - lo], [hi - fm["RMSE"]]], fmt="none", c="k", capsize=4)
ax.set_xlabel("Test RMSE (düşük daha iyi)"); ax.set_title("FD001 — literatür karşılaştırması (siyah çubuk: %95 güven aralığı)")
plt.tight_layout(); plt.show()

> 📝 **Rapor için:** Final modelin hangi çalışmaların önünde/gerisinde olduğunu, güven aralığını da dikkate alarak yazın. Güncel derin öğrenme modellerinin (DCNN ve sonrası, ~11–13) gerisinde kalmak beklenen bir sonuç. Klasik bir modelin avantajlarını da tartışın: GPU gerektirmemesi, saniyeler içinde eğitilmesi, ~1 MB boyut ve SHAP ile açıklanabilirlik.

### 12.9 Güçlü ve zayıf yönler — otomatik özet

In [ ]:
crit_recall = (len(crit) - len(missed)) / max(len(crit), 1) * 100
late = res[res["hata"] > 10]
over125 = res[res["RUL_true"] > CAP]
facts = [
    f"Test RMSE {fm['RMSE']:.2f} (%95 GA [{lo:.2f}, {hi:.2f}]); baseline LR {bl['RMSE']:.2f} → %{(bl['RMSE']-fm['RMSE'])/bl['RMSE']*100:.0f} iyileşme.",
    f"CV tahmini {cv_m:.2f} ± {cv_s:.2f}; test sonucu {'CV ile tutarlı' if inside else 'CV aralığının dışında'}.",
    f"Arızaya yakın motorlarda (RUL ≤ 50) RMSE {fm['RMSE (RUL≤50)']:.2f}.",
    f"Kritik motorları yakalama oranı (recall) %{crit_recall:.0f} ({len(crit) - len(missed)}/{len(crit)}).",
    f"Ömrü 10 döngüden fazla fazla tahmin edilen (geç tahmin) motor: {len(late)} / 100.",
    f"Gerçek RUL'u {CAP}'in üzerinde olan {len(over125)} motorda ortalama hata {over125['hata'].mean():+.1f} (tasarım gereği temkinli).",
    f"Ortalama hata {res['hata'].mean():+.1f} döngü (" + ("yaklaşık dengeli: sistematik bir sapma yok" if abs(res['hata'].mean()) < 2
        else "temkinli: ömrü eksik tahmin etme eğilimi" if res['hata'].mean() < 0 else "iyimser: ömrü fazla tahmin etme eğilimi") + ").",
    f"Model {mdl.__class__.__name__}, pencere {WINDOW}, dosya boyutu {os.path.getsize('final_model.joblib')/1e6:.2f} MB.",
]
for f in facts:
    print("•", f)

> 📝 **Rapor için — güçlü ve zayıf yönler paragrafı:** Yukarıdaki maddeleri kullanarak yazın. Önerilen yapı:
> - **Güçlü yönler:** Baseline'a göre iyileşme · literatürdeki erken derin öğrenme yöntemleriyle karşılaştırılabilir isabet · arızaya yakın bölgede düşük hata · hızlı ve küçük model · SHAP ile açıklanabilir · sızıntısız ve tekrarlanabilir deney tasarımı.
> - **Zayıf yönler:** 125 üzerindeki RUL'lar ayırt edilemiyor · kısa geçmişte belirsizlik (12.6) · güncel derin öğrenme modellerinin gerisinde · yalnızca FD001 (tek çalışma koşulu, tek arıza tipi) · simülasyon verisi.
> - **Gelecek çalışmalar:** FD002–FD004'e genişletme (çoklu çalışma koşulu) · tahmin aralığı (ör. quantile regression ile "RUL 40 ± 12") · LSTM/1D-CNN ile karşılaştırma · NASA skorunu doğrudan optimize eden asimetrik kayıp fonksiyonu.

### Sonuçları kaydet

In [ ]:
res.to_csv("results_stage12_test_predictions.csv", index=False)
rob.to_csv("results_stage12_robustness.csv", index=False)
cfg = json.load(open("project_config.json")) if os.path.exists("project_config.json") else {}
cfg.update({"test_RMSE": float(fm["RMSE"]), "test_RMSE_CI95": [float(lo), float(hi)], "test_NASA": float(fm["NASA"]),
            "test_MAE": float(fm["MAE"]), "test_R2": float(fm["R2"]), "test_RMSE_near50": float(fm["RMSE (RUL≤50)"]),
            "test_critical_recall_pct": float(crit_recall), "risk_thresholds": {"kritik": CRIT, "bakim_planla": PLAN}})
json.dump(cfg, open("project_config.json", "w"), indent=2, ensure_ascii=False, default=str)
print("Kaydedildi: results_stage12_test_metrics.csv, results_stage12_test_predictions.csv, results_stage12_robustness.csv, project_config.json")